# String Pot Analysis

This notebook is a standalone analysis workspace for the homemade string potentiometer test data. It stays separate from the main `bodaqs_analysis` package and works directly from a logger CSV plus the crank-drive geometry summary in this folder.

Workflow:
- load a CSV and select the time plus raw ADC columns
- apply a configurable cubic correction in corrected-count units
- unwrap the corrected signal with a configurable wrap span
- estimate the cycle period from the measured signal
- build the ideal string displacement profile from the crank geometry
- fit only scale and time shift between the measured and ideal profiles
- inspect residuals and summary metrics

In [33]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import optimize, signal

pio.renderers.default = 'notebook_connected'

In [34]:
# Parameters

analysis_dir = Path('.')
csv_path = analysis_dir / '500rpm.CSV'

time_column = 'timestamp'
raw_adc_column = 'i2c_stringpot_raw [counts]'
time_format = '%H:%M:%S.%f'
analysis_start_time_s = 2
analysis_end_time_s = 8
wrap_span_counts = 4095.0

# Cubic correction in corrected-count units.
c0 = 0.0
c1 = 1
c2 = 0
c3 = 0.0

# Crank-drive geometry in mm.
crank_pin_offset_r_mm = 68.2
entry_offset_l_mm = 450.0

# Period estimation and fitting controls.
sample_rate_hz_override = None
driving_speed_rpm_override = None
period_search_min_rpm = 450.0
period_search_max_rpm = 1600.0
fit_shift_search_fraction_of_period = 0.5
minimum_valid_fraction_for_fit = 0.80
show_measured_displacement_datapoints = True
phase_bin_count = 72
phase_harmonic_order = 3

In [35]:
def _require_column(df: pd.DataFrame, column_name: str) -> None:
    if column_name not in df.columns:
        raise KeyError(f"Column '{column_name}' not found. Available columns: {list(df.columns)}")


def load_time_seconds(series: pd.Series, time_format: str | None = None) -> np.ndarray:
    numeric = pd.to_numeric(series, errors='coerce')
    numeric_values = numeric.to_numpy(dtype=float)
    if np.isfinite(numeric_values).any() and np.isfinite(numeric_values).sum() == len(series):
        first_idx = np.flatnonzero(np.isfinite(numeric_values))[0]
        return numeric_values - numeric_values[first_idx]

    parsed = pd.to_datetime(series, format=time_format, errors='coerce')
    valid = parsed.notna().to_numpy(dtype=bool)
    if not valid.any():
        raise ValueError('Could not convert the time column to a relative-seconds vector.')

    base_time = parsed[valid].iloc[0]
    rel_s = (parsed - base_time).dt.total_seconds().to_numpy(dtype=float)
    return rel_s


def apply_polynomial_correction(raw_counts: np.ndarray, coeffs: tuple[float, float, float, float]) -> np.ndarray:
    c0_, c1_, c2_, c3_ = coeffs
    raw_counts = np.asarray(raw_counts, dtype=float)
    return c0_ + c1_ * raw_counts + c2_ * raw_counts**2 + c3_ * raw_counts**3


def unwrap_periodic_signal(values: np.ndarray, wrap_span: float) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    out = values.copy()
    valid_idx = np.flatnonzero(np.isfinite(values))
    if valid_idx.size == 0:
        return out

    offset = 0.0
    prev_unwrapped = values[valid_idx[0]]
    out[valid_idx[0]] = prev_unwrapped

    for pos in range(1, valid_idx.size):
        idx = valid_idx[pos]
        current = values[idx]

        candidates = current + wrap_span * np.array([-1.0, 0.0, 1.0]) + offset
        best = candidates[np.argmin(np.abs(candidates - prev_unwrapped))]

        if best - prev_unwrapped > (wrap_span / 2.0):
            best -= wrap_span
        elif best - prev_unwrapped < -(wrap_span / 2.0):
            best += wrap_span

        offset = best - current
        out[idx] = best
        prev_unwrapped = best

    return out


def _safe_savgol_window(length: int, requested: int, poly_order: int) -> int | None:
    if length <= poly_order:
        return None
    window = min(requested, length if length % 2 == 1 else length - 1)
    if window <= poly_order:
        window = poly_order + 2
        if window % 2 == 0:
            window += 1
    if window > length:
        window = length if length % 2 == 1 else length - 1
    if window <= poly_order or window < 3:
        return None
    return window


def smooth_series(values: np.ndarray, requested_window: int, poly_order: int) -> np.ndarray:
    y = np.asarray(values, dtype=float)
    valid = np.isfinite(y)
    if valid.sum() < poly_order + 2:
        return y.copy()

    filled = pd.Series(y).interpolate(limit_direction='both').to_numpy(dtype=float)
    window = _safe_savgol_window(len(filled), requested_window, poly_order)
    if window is None:
        return filled
    return signal.savgol_filter(filled, window_length=window, polyorder=poly_order, mode='interp')


def estimate_sample_rate_hz(t_s: np.ndarray, sample_rate_hz_override: float | None = None) -> float:
    if sample_rate_hz_override is not None:
        sample_rate_hz = float(sample_rate_hz_override)
        if not np.isfinite(sample_rate_hz) or sample_rate_hz <= 0.0:
            raise ValueError('sample_rate_hz_override must be positive when provided.')
        return sample_rate_hz

    t_s = np.asarray(t_s, dtype=float)
    dt = np.diff(t_s)
    dt = dt[np.isfinite(dt) & (dt > 0.0)]
    if dt.size == 0:
        raise ValueError('Could not infer a positive sample interval from the time vector.')

    dt_median = float(np.median(dt))
    return 1.0 / dt_median


def estimate_period_seconds(
    t_s: np.ndarray,
    wrapped_counts: np.ndarray,
    wrap_span: float,
    min_rpm: float,
    max_rpm: float,
    sample_rate_hz_override: float | None = None,
) -> dict:
    mask = np.isfinite(t_s) & np.isfinite(wrapped_counts)
    if mask.sum() < 10:
        raise ValueError('Need at least 10 valid samples to estimate period.')

    t_valid = np.asarray(t_s[mask], dtype=float)
    y_valid = np.asarray(wrapped_counts[mask], dtype=float)

    order = np.argsort(t_valid)
    t_valid = t_valid[order]
    y_valid = y_valid[order]

    sample_rate_hz = estimate_sample_rate_hz(t_valid, sample_rate_hz_override=sample_rate_hz_override)
    if not np.isfinite(min_rpm) or not np.isfinite(max_rpm) or min_rpm <= 0.0 or max_rpm <= min_rpm:
        raise ValueError('RPM search bounds must be finite and satisfy 0 < min_rpm < max_rpm.')

    phase_rad = 2.0 * np.pi * y_valid / float(wrap_span)
    circular_cos = np.cos(phase_rad)
    circular_sin = np.sin(phase_rad)
    circular_cos_centered = circular_cos - np.mean(circular_cos)
    circular_sin_centered = circular_sin - np.mean(circular_sin)

    autocorr = signal.correlate(circular_cos_centered, circular_cos_centered, mode='full')
    autocorr += signal.correlate(circular_sin_centered, circular_sin_centered, mode='full')
    autocorr = autocorr[autocorr.size // 2 :]
    lag_samples = np.arange(autocorr.size, dtype=float)
    lag_s = lag_samples / sample_rate_hz

    min_period_s = 60.0 / float(max_rpm)
    max_period_s = 60.0 / float(min_rpm)
    search_mask = (lag_s >= min_period_s) & (lag_s <= max_period_s)
    if not np.any(search_mask):
        raise ValueError('RPM search bounds do not map to any valid lag samples.')

    search_idx = np.flatnonzero(search_mask)
    autocorr_search = autocorr[search_idx]
    peak_offsets, _ = signal.find_peaks(autocorr_search)
    if peak_offsets.size == 0:
        best_idx = int(search_idx[np.argmax(autocorr_search)])
        candidate_periods = np.asarray([lag_s[best_idx]], dtype=float)
        candidate_strengths = np.asarray([autocorr[best_idx]], dtype=float)
    else:
        peak_idx = search_idx[peak_offsets]
        peak_strengths = autocorr[peak_idx]
        order = np.argsort(peak_strengths)[::-1]
        peak_idx = peak_idx[order]
        peak_strengths = peak_strengths[order]
        candidate_periods = lag_s[peak_idx]
        candidate_strengths = peak_strengths

    period_s = float(candidate_periods[0])
    if not np.isfinite(period_s) or period_s <= 0.0:
        raise ValueError('Estimated period is not positive.')

    nperseg = min(len(y_valid), 4096)
    if nperseg < 8:
        raise ValueError('Not enough valid samples for spectral cross-check.')
    freq_hz, psd_cos = signal.welch(circular_cos_centered, fs=sample_rate_hz, nperseg=nperseg)
    _, psd_sin = signal.welch(circular_sin_centered, fs=sample_rate_hz, nperseg=nperseg)
    psd_total = psd_cos + psd_sin
    search_freq_mask = (freq_hz >= 1.0 / max_period_s) & (freq_hz <= 1.0 / min_period_s)
    dominant_frequency_hz = float(freq_hz[search_freq_mask][np.argmax(psd_total[search_freq_mask])])

    return {
        'period_s': period_s,
        'method': 'circular-autocorrelation',
        'sample_rate_hz': sample_rate_hz,
        't_valid': t_valid,
        'wrapped_counts_valid': y_valid,
        'circular_cos': circular_cos,
        'circular_sin': circular_sin,
        'autocorr': autocorr,
        'lag_s': lag_s,
        'search_mask': search_mask,
        'candidate_periods_s': np.asarray(candidate_periods, dtype=float),
        'candidate_strengths': np.asarray(candidate_strengths, dtype=float),
        'dominant_frequency_hz': dominant_frequency_hz,
    }


def ideal_string_displacement_mm(theta_rad: np.ndarray, r_mm: float, l_mm: float) -> np.ndarray:
    theta_rad = np.asarray(theta_rad, dtype=float)
    s_mm = np.sqrt(l_mm**2 + r_mm**2 - 2.0 * l_mm * r_mm * np.cos(theta_rad))
    return s_mm - (l_mm - r_mm)


def ideal_profile_from_time(t_s: np.ndarray, period_s: float, r_mm: float, l_mm: float) -> np.ndarray:
    theta = 2.0 * np.pi * np.asarray(t_s, dtype=float) / period_s
    return ideal_string_displacement_mm(theta, r_mm=r_mm, l_mm=l_mm)


def shift_signal_in_time(t_ref: np.ndarray, t_src: np.ndarray, y_src: np.ndarray, shift_s: float) -> np.ndarray:
    shifted_time = np.asarray(t_src, dtype=float) + float(shift_s)
    return np.interp(
        np.asarray(t_ref, dtype=float),
        shifted_time,
        np.asarray(y_src, dtype=float),
        left=np.nan,
        right=np.nan,
    )


def refine_extrema_times_quadratic(
    t_s: np.ndarray,
    smoothed_values: np.ndarray,
    extrema_idx: np.ndarray,
    sample_rate_hz: float,
) -> tuple[np.ndarray, np.ndarray]:
    t_s = np.asarray(t_s, dtype=float)
    smoothed_values = np.asarray(smoothed_values, dtype=float)
    extrema_idx = np.asarray(extrema_idx, dtype=int)
    refined_times = t_s[extrema_idx].astype(float).copy()
    refined_values = smoothed_values[extrema_idx].astype(float).copy()

    for j, idx in enumerate(extrema_idx):
        if 0 < idx < (len(smoothed_values) - 1):
            y_m1 = float(smoothed_values[idx - 1])
            y_0 = float(smoothed_values[idx])
            y_p1 = float(smoothed_values[idx + 1])
            denom = y_m1 - 2.0 * y_0 + y_p1
            if np.isfinite(denom) and abs(denom) > 1e-12:
                offset_samples = 0.5 * (y_m1 - y_p1) / denom
                offset_samples = float(np.clip(offset_samples, -1.0, 1.0))
                refined_times[j] = t_s[idx] + offset_samples / float(sample_rate_hz)
                refined_values[j] = y_0 - 0.25 * (y_m1 - y_p1) * offset_samples

    return refined_times, refined_values


def detect_principal_extrema_times(
    t_s: np.ndarray,
    values: np.ndarray,
    period_s: float,
    sample_rate_hz: float,
) -> dict:
    if not np.isfinite(period_s) or period_s <= 0.0:
        raise ValueError('period_s must be positive for extrema detection.')
    if not np.isfinite(sample_rate_hz) or sample_rate_hz <= 0.0:
        raise ValueError('sample_rate_hz must be positive for extrema detection.')

    t_s = np.asarray(t_s, dtype=float)
    values = np.asarray(values, dtype=float)
    mask = np.isfinite(t_s) & np.isfinite(values)
    if mask.sum() < 10:
        raise ValueError('Need at least 10 valid samples to detect extrema.')

    t_valid = t_s[mask]
    y_valid = values[mask]
    order = np.argsort(t_valid)
    t_valid = t_valid[order]
    y_valid = y_valid[order]

    requested_window = int(np.round(max(5.0, 0.15 * period_s * sample_rate_hz)))
    if requested_window % 2 == 0:
        requested_window += 1
    smoothed = smooth_series(y_valid, requested_window=requested_window, poly_order=3)

    span = float(np.nanmax(smoothed) - np.nanmin(smoothed))
    prominence = max(1.0, 0.15 * span)
    min_distance = max(1, int(np.floor(0.60 * period_s * sample_rate_hz)))

    peak_idx, _ = signal.find_peaks(smoothed, distance=min_distance, prominence=prominence)
    trough_idx, _ = signal.find_peaks(-smoothed, distance=min_distance, prominence=prominence)

    if peak_idx.size < 2 or trough_idx.size < 2:
        raise ValueError('Could not detect enough principal extrema for time-shift alignment.')

    peak_times_s, peak_values = refine_extrema_times_quadratic(
        t_valid,
        smoothed,
        peak_idx,
        sample_rate_hz=sample_rate_hz,
    )
    trough_times_s, trough_values = refine_extrema_times_quadratic(
        t_valid,
        -smoothed,
        trough_idx,
        sample_rate_hz=sample_rate_hz,
    )
    trough_values = -trough_values

    return {
        'peak_times_s': peak_times_s,
        'trough_times_s': trough_times_s,
        'peak_times_sample_s': t_valid[peak_idx],
        'trough_times_sample_s': t_valid[trough_idx],
        'peak_values': peak_values,
        'trough_values': trough_values,
        'peak_values_sample': y_valid[peak_idx],
        'trough_values_sample': y_valid[trough_idx],
        'smoothed_signal': smoothed,
        't_valid': t_valid,
        'signal_valid': y_valid,
        'peak_idx': peak_idx,
        'trough_idx': trough_idx,
    }


def summarize_cycle_period_from_extrema(extrema_info: dict) -> dict:
    peak_periods_s = np.diff(np.asarray(extrema_info['peak_times_s'], dtype=float))
    trough_periods_s = np.diff(np.asarray(extrema_info['trough_times_s'], dtype=float))
    period_samples = []
    if peak_periods_s.size:
        period_samples.append(peak_periods_s)
    if trough_periods_s.size:
        period_samples.append(trough_periods_s)
    if not period_samples:
        raise ValueError('Could not compute cycle periods from detected extrema.')

    all_periods_s = np.concatenate(period_samples)
    all_periods_s = all_periods_s[np.isfinite(all_periods_s) & (all_periods_s > 0.0)]
    if all_periods_s.size == 0:
        raise ValueError('No valid positive cycle periods were found in the detected extrema.')

    return {
        'period_s': float(np.mean(all_periods_s)),
        'period_std_s': float(np.std(all_periods_s)),
        'peak_periods_s': peak_periods_s,
        'trough_periods_s': trough_periods_s,
        'all_periods_s': all_periods_s,
    }


def periodic_distance_to_grid(times_s: np.ndarray, period_s: float, phase_offset_s: float) -> np.ndarray:
    times_s = np.asarray(times_s, dtype=float)
    wrapped = ((times_s - phase_offset_s + 0.5 * period_s) % period_s) - 0.5 * period_s
    return np.abs(wrapped)


def _extrema_alignment_mse(
    period_s: float,
    shift_s: float,
    peak_times_s: np.ndarray,
    trough_times_s: np.ndarray,
    fit_start_s: float,
    fit_end_s: float,
    minimum_valid_extrema: int,
) -> float:
    if not np.isfinite(period_s) or period_s <= 0.0 or not np.isfinite(shift_s):
        return np.inf

    shifted_peak_times = np.asarray(peak_times_s, dtype=float) + float(shift_s)
    shifted_trough_times = np.asarray(trough_times_s, dtype=float) + float(shift_s)

    valid_peak = (shifted_peak_times >= fit_start_s) & (shifted_peak_times <= fit_end_s)
    valid_trough = (shifted_trough_times >= fit_start_s) & (shifted_trough_times <= fit_end_s)
    valid_count = int(np.count_nonzero(valid_peak) + np.count_nonzero(valid_trough))
    if valid_count < int(minimum_valid_extrema):
        return np.inf

    peak_dist = periodic_distance_to_grid(
        shifted_peak_times[valid_peak],
        period_s=period_s,
        phase_offset_s=0.5 * period_s,
    )
    trough_dist = periodic_distance_to_grid(
        shifted_trough_times[valid_trough],
        period_s=period_s,
        phase_offset_s=0.0,
    )
    distances = np.concatenate([peak_dist, trough_dist])
    if distances.size == 0 or not np.all(np.isfinite(distances)):
        return np.inf
    return float(np.mean(distances**2))


def _search_extrema_alignment_global(
    period_lower_s: float,
    period_upper_s: float,
    shift_bound_s: float,
    peak_times_s: np.ndarray,
    trough_times_s: np.ndarray,
    fit_start_s: float,
    fit_end_s: float,
    minimum_valid_extrema: int,
    sample_rate_hz: float,
) -> dict:
    period_span_s = float(period_upper_s - period_lower_s)
    period_grid_points = int(np.clip(np.ceil(period_span_s / max(1e-6, 0.05 / sample_rate_hz)) + 1, 81, 241))
    shift_grid_points = int(np.clip(np.ceil((2.0 * shift_bound_s) / max(1e-6, 0.05 / sample_rate_hz)) + 1, 401, 1601))

    period_candidates = np.linspace(period_lower_s, period_upper_s, period_grid_points, dtype=float)
    shift_candidates = np.linspace(-shift_bound_s, shift_bound_s, shift_grid_points, dtype=float)

    best_period_s = np.nan
    best_shift_s = np.nan
    best_mse = np.inf
    for period_s in period_candidates:
        objective_values = np.asarray(
            [
                _extrema_alignment_mse(
                    float(period_s),
                    float(shift_s),
                    peak_times_s,
                    trough_times_s,
                    fit_start_s,
                    fit_end_s,
                    minimum_valid_extrema,
                )
                for shift_s in shift_candidates
            ],
            dtype=float,
        )
        best_idx = int(np.argmin(objective_values))
        best_value = float(objective_values[best_idx])
        if np.isfinite(best_value) and best_value < best_mse:
            best_mse = best_value
            best_period_s = float(period_s)
            best_shift_s = float(shift_candidates[best_idx])

    if not np.isfinite(best_mse):
        raise RuntimeError('Global extrema-alignment search did not find a finite solution.')

    return {
        'period_s': best_period_s,
        'shift_s': best_shift_s,
        'mse': best_mse,
        'period_grid_points': period_grid_points,
        'shift_grid_points': shift_grid_points,
        'period_candidates_s': period_candidates,
        'shift_candidates_s': shift_candidates,
    }


def fit_scale_and_shift(
    t_s: np.ndarray,
    measured_counts_zeroed: np.ndarray,
    r_mm: float,
    l_mm: float,
    period_guess_s: float,
    shift_fraction: float,
    min_valid_fraction: float,
    sample_rate_hz: float,
) -> dict:
    mask = np.isfinite(t_s) & np.isfinite(measured_counts_zeroed)
    if mask.sum() < 10:
        raise ValueError('Need at least 10 valid samples to fit period, scale, and time shift.')

    t_fit = np.asarray(t_s[mask], dtype=float)
    measured_fit = np.asarray(measured_counts_zeroed[mask], dtype=float)
    fit_start_s = float(np.nanmin(t_fit))
    fit_end_s = float(np.nanmax(t_fit))

    extrema_info = detect_principal_extrema_times(
        t_fit,
        measured_fit,
        period_s=period_guess_s,
        sample_rate_hz=sample_rate_hz,
    )
    cycle_period_info = summarize_cycle_period_from_extrema(extrema_info)
    refined_period_s = float(cycle_period_info['period_s'])
    refined_period_std_s = float(cycle_period_info['period_std_s'])

    shift_bound = abs(float(shift_fraction)) * refined_period_s
    minimum_valid_samples = max(10, int(np.ceil(min_valid_fraction * len(t_fit))))
    total_extrema_count = len(extrema_info['peak_times_s']) + len(extrema_info['trough_times_s'])
    minimum_valid_extrema = max(4, int(np.ceil(min_valid_fraction * total_extrema_count)))

    period_half_width_s = max(
        5.0 * refined_period_std_s,
        2.0 / float(sample_rate_hz),
        0.02 * refined_period_s,
        abs(refined_period_s - float(period_guess_s)) + (1.0 / float(sample_rate_hz)),
    )
    period_half_width_s = min(period_half_width_s, 0.10 * refined_period_s)
    period_lower_s = max(1e-9, refined_period_s - period_half_width_s)
    period_upper_s = refined_period_s + period_half_width_s

    global_search = _search_extrema_alignment_global(
        period_lower_s=period_lower_s,
        period_upper_s=period_upper_s,
        shift_bound_s=shift_bound,
        peak_times_s=np.asarray(extrema_info['peak_times_s'], dtype=float),
        trough_times_s=np.asarray(extrema_info['trough_times_s'], dtype=float),
        fit_start_s=fit_start_s,
        fit_end_s=fit_end_s,
        minimum_valid_extrema=minimum_valid_extrema,
        sample_rate_hz=float(sample_rate_hz),
    )

    period_candidates = np.asarray(global_search['period_candidates_s'], dtype=float)
    shift_candidates = np.asarray(global_search['shift_candidates_s'], dtype=float)
    period_step_s = float(period_candidates[1] - period_candidates[0]) if period_candidates.size > 1 else 0.5 * (period_upper_s - period_lower_s)
    shift_step_s = float(shift_candidates[1] - shift_candidates[0]) if shift_candidates.size > 1 else shift_bound

    local_period_lower = max(period_lower_s, float(global_search['period_s']) - period_step_s)
    local_period_upper = min(period_upper_s, float(global_search['period_s']) + period_step_s)
    local_shift_lower = max(-shift_bound, float(global_search['shift_s']) - shift_step_s)
    local_shift_upper = min(shift_bound, float(global_search['shift_s']) + shift_step_s)

    def objective(params: np.ndarray) -> float:
        return _extrema_alignment_mse(
            float(params[0]),
            float(params[1]),
            extrema_info['peak_times_s'],
            extrema_info['trough_times_s'],
            fit_start_s,
            fit_end_s,
            minimum_valid_extrema,
        )

    opt = optimize.minimize(
        objective,
        x0=np.asarray([float(global_search['period_s']), float(global_search['shift_s'])], dtype=float),
        method='Powell',
        bounds=[
            (local_period_lower, local_period_upper),
            (local_shift_lower, local_shift_upper),
        ],
        options={
            'xtol': max(refined_period_s * 1e-7, 1e-10),
            'ftol': 1e-14,
            'maxiter': 300,
        },
    )
    if not opt.success:
        raise RuntimeError(f'Local period/shift refinement did not converge: {opt.message}')

    best_period_s = float(opt.x[0])
    best_shift_s = float(opt.x[1])

    shifted_peak_times = np.asarray(extrema_info['peak_times_s'], dtype=float) + best_shift_s
    shifted_trough_times = np.asarray(extrema_info['trough_times_s'], dtype=float) + best_shift_s
    valid_peak = (shifted_peak_times >= fit_start_s) & (shifted_peak_times <= fit_end_s)
    valid_trough = (shifted_trough_times >= fit_start_s) & (shifted_trough_times <= fit_end_s)
    peak_dist = periodic_distance_to_grid(
        shifted_peak_times[valid_peak],
        period_s=best_period_s,
        phase_offset_s=0.5 * best_period_s,
    )
    trough_dist = periodic_distance_to_grid(
        shifted_trough_times[valid_trough],
        period_s=best_period_s,
        phase_offset_s=0.0,
    )
    extrema_alignment_distances_s = np.concatenate([peak_dist, trough_dist])

    ideal_mm = ideal_profile_from_time(
        t_s,
        period_s=best_period_s,
        r_mm=r_mm,
        l_mm=l_mm,
    )

    shifted_full = shift_signal_in_time(t_s, t_s, measured_counts_zeroed, best_shift_s)
    valid_full = np.isfinite(shifted_full) & np.isfinite(ideal_mm)
    if int(np.count_nonzero(valid_full)) < minimum_valid_samples:
        raise RuntimeError('Best-fit shift does not leave enough valid samples for residual analysis.')

    y_full = shifted_full[valid_full]
    x_full = ideal_mm[valid_full]
    measured_peak_full = float(np.nanmax(y_full))
    ideal_peak_full = float(np.nanmax(x_full))
    if measured_peak_full <= 0.0 or not np.isfinite(measured_peak_full) or not np.isfinite(ideal_peak_full):
        raise RuntimeError('Best-fit shifted signal does not have a valid positive peak for peak-matched scaling.')

    best_scale_mm_per_count = float(ideal_peak_full / measured_peak_full)
    scaled_full_mm = best_scale_mm_per_count * shifted_full
    residuals_full_mm = scaled_full_mm - ideal_mm
    rmse_mm = float(np.sqrt(np.nanmean(residuals_full_mm[valid_full] ** 2)))

    return {
        'period_s': best_period_s,
        'frequency_hz': float(1.0 / best_period_s),
        'speed_rpm': float(60.0 / best_period_s),
        'period_initial_guess_s': float(period_guess_s),
        'refined_cycle_period_s': refined_period_s,
        'refined_cycle_period_std_s': refined_period_std_s,
        'period_search_bounds_s': (float(period_lower_s), float(period_upper_s)),
        'period_shift_search_method': 'global_grid_then_local_refine',
        'period_grid_points': int(global_search['period_grid_points']),
        'shift_grid_points': int(global_search['shift_grid_points']),
        'shift_s': best_shift_s,
        'scale_mm_per_count': best_scale_mm_per_count,
        'ideal_displacement_mm': ideal_mm,
        'scaled_series_mm': scaled_full_mm,
        'shifted_counts': shifted_full,
        'residuals_mm': residuals_full_mm,
        'valid_mask': valid_full,
        'measured_peak_count': measured_peak_full,
        'ideal_peak_mm': ideal_peak_full,
        'rmse_mm': rmse_mm,
        'optimizer_result': opt,
        'global_search_result': global_search,
        'shift_objective': 'extrema_alignment_period_and_shift',
        'extrema_alignment_rmse_s': float(np.sqrt(np.mean(extrema_alignment_distances_s**2))),
        'extrema_alignment_peak_times_s': extrema_info['peak_times_s'],
        'extrema_alignment_trough_times_s': extrema_info['trough_times_s'],
        'extrema_detection': extrema_info,
    }

def summarize_residual_vs_phase(
    t_s: np.ndarray,
    residual_mm: np.ndarray,
    valid_mask: np.ndarray,
    period_s: float,
    bin_count: int,
    max_harmonic: int,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    if not np.isfinite(period_s) or period_s <= 0.0:
        raise ValueError('period_s must be positive for phase-residual analysis.')
    if int(bin_count) < 8:
        raise ValueError('bin_count must be at least 8 for phase-residual analysis.')
    if int(max_harmonic) < 1:
        raise ValueError('max_harmonic must be at least 1 for phase-residual analysis.')

    t_s = np.asarray(t_s, dtype=float)
    residual_mm = np.asarray(residual_mm, dtype=float)
    valid_mask = np.asarray(valid_mask, dtype=bool)
    mask = valid_mask & np.isfinite(t_s) & np.isfinite(residual_mm)
    if mask.sum() < int(bin_count):
        raise ValueError('Not enough valid samples for phase-residual analysis.')

    t_valid = t_s[mask]
    residual_valid = residual_mm[mask]
    phase_fraction = (t_valid / float(period_s)) % 1.0
    phase_deg = 360.0 * phase_fraction
    phase_rad = 2.0 * np.pi * phase_fraction

    bin_edges_deg = np.linspace(0.0, 360.0, int(bin_count) + 1)
    bin_idx = np.digitize(phase_deg, bin_edges_deg, right=False) - 1
    bin_idx = np.clip(bin_idx, 0, int(bin_count) - 1)

    phase_rows: list[dict] = []
    for idx in range(int(bin_count)):
        in_bin = bin_idx == idx
        center_deg = 0.5 * (bin_edges_deg[idx] + bin_edges_deg[idx + 1])
        if np.any(in_bin):
            values = residual_valid[in_bin]
            mean_residual = float(np.mean(values))
            std_residual = float(np.std(values))
            rms_residual = float(np.sqrt(np.mean(values**2)))
        else:
            mean_residual = np.nan
            std_residual = np.nan
            rms_residual = np.nan
        phase_rows.append(
            {
                'phase_center_deg': center_deg,
                'phase_start_deg': float(bin_edges_deg[idx]),
                'phase_end_deg': float(bin_edges_deg[idx + 1]),
                'sample_count': int(np.count_nonzero(in_bin)),
                'mean_residual_mm': mean_residual,
                'std_residual_mm': std_residual,
                'rms_residual_mm': rms_residual,
                'mean_plus_std_mm': mean_residual + std_residual if np.isfinite(mean_residual) and np.isfinite(std_residual) else np.nan,
                'mean_minus_std_mm': mean_residual - std_residual if np.isfinite(mean_residual) and np.isfinite(std_residual) else np.nan,
            }
        )
    phase_bins_df = pd.DataFrame(phase_rows)

    design_columns = [np.ones_like(phase_rad)]
    for harmonic in range(1, int(max_harmonic) + 1):
        design_columns.append(np.cos(harmonic * phase_rad))
        design_columns.append(np.sin(harmonic * phase_rad))
    design_matrix = np.column_stack(design_columns)
    coeffs, *_ = np.linalg.lstsq(design_matrix, residual_valid, rcond=None)
    harmonic_rows: list[dict] = [
        {
            'harmonic': 0,
            'amplitude_mm': float(coeffs[0]),
            'phase_offset_deg': np.nan,
            'cos_coeff_mm': float(coeffs[0]),
            'sin_coeff_mm': np.nan,
        }
    ]
    for harmonic in range(1, int(max_harmonic) + 1):
        cos_coeff = float(coeffs[2 * harmonic - 1])
        sin_coeff = float(coeffs[2 * harmonic])
        harmonic_rows.append(
            {
                'harmonic': harmonic,
                'amplitude_mm': float(np.hypot(cos_coeff, sin_coeff)),
                'phase_offset_deg': float(np.mod(np.degrees(np.arctan2(sin_coeff, cos_coeff)), 360.0)),
                'cos_coeff_mm': cos_coeff,
                'sin_coeff_mm': sin_coeff,
            }
        )
    harmonics_df = pd.DataFrame(harmonic_rows)

    return phase_bins_df, harmonics_df, {
        'phase_bins_df': phase_bins_df,
        'harmonics_df': harmonics_df,
        'phase_deg': phase_deg,
        'phase_rad': phase_rad,
        'valid_residual_mm': residual_valid,
        'bin_count': int(bin_count),
    }


In [36]:
df = pd.read_csv(csv_path)
_require_column(df, time_column)
_require_column(df, raw_adc_column)

raw_time = df[time_column]
raw_adc_counts = pd.to_numeric(df[raw_adc_column], errors='coerce').to_numpy(dtype=float)
time_s_full = load_time_seconds(raw_time, time_format=time_format)

window_mask = np.isfinite(time_s_full)
if analysis_start_time_s is not None:
    window_mask &= time_s_full >= float(analysis_start_time_s)
if analysis_end_time_s is not None:
    window_mask &= time_s_full <= float(analysis_end_time_s)
if not np.any(window_mask):
    raise ValueError('The requested analysis time window does not contain any valid samples.')

df_window = df.loc[window_mask].copy()
time_s = time_s_full[window_mask]
raw_adc_counts = raw_adc_counts[window_mask]

coeffs = (c0, c1, c2, c3)
corrected_counts = apply_polynomial_correction(raw_adc_counts, coeffs)
unwrapped_counts = unwrap_periodic_signal(
    corrected_counts,
    wrap_span=wrap_span_counts,
)
wrap_count = np.rint((unwrapped_counts - corrected_counts) / wrap_span_counts)
unwrapped_counts_zeroed = unwrapped_counts - np.nanmin(unwrapped_counts)

processed_df = pd.DataFrame(
    {
        'time_s': time_s,
        'raw_adc_counts': raw_adc_counts,
        'corrected_counts': corrected_counts,
        'wrap_count': wrap_count,
        'unwrapped_counts': unwrapped_counts,
        'unwrapped_counts_zeroed': unwrapped_counts_zeroed,
    }
)

display(df_window.head())
display(processed_df.head())

,sample_id,timestamp,i2c_stringpot_raw [counts],i2c_stringpot [mm],mark
1000,1000,00:50:32.093,699.0,7.080969,0.0
1001,1001,00:50:32.095,958.0,3.913489,0.0
1002,1002,00:50:32.097,1286.0,-0.097837,0.0
1003,1003,00:50:32.099,1568.0,-3.546599,0.0
1004,1004,00:50:32.101,1815.0,-6.567323,0.0


,time_s,raw_adc_counts,corrected_counts,wrap_count,unwrapped_counts,unwrapped_counts_zeroed
0,2.000,699.0,699.0,0.0,699.0,5001.0
1,2.002,958.0,958.0,0.0,958.0,5260.0
2,2.004,1286.0,1286.0,0.0,1286.0,5588.0
3,2.006,1568.0,1568.0,0.0,1568.0,5870.0
4,2.008,1815.0,1815.0,0.0,1815.0,6117.0


In [37]:
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=('Raw ADC signal', 'Polynomial-corrected signal', 'Unwrapped corrected signal', 'Wraps'),
)
fig.add_trace(
    go.Scatter(x=processed_df['time_s'], y=processed_df['raw_adc_counts'], mode='lines', name='Raw counts', line=dict(width=1.0)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=processed_df['time_s'], y=processed_df['corrected_counts'], mode='lines', name='Corrected counts', line=dict(width=1.0, color='orange')),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=processed_df['time_s'], y=processed_df['unwrapped_counts'], mode='lines', name='Unwrapped counts', line=dict(width=1.0, color='green')),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(x=processed_df['time_s'], y=processed_df['wrap_count'], mode='lines', name='Wrap count', line=dict(width=1.0, color='purple')),
    row=4,
    col=1,
)
fig.update_yaxes(title_text='Raw counts', row=1, col=1)
fig.update_yaxes(title_text='Corrected counts', row=2, col=1)
fig.update_yaxes(title_text='Unwrapped counts', row=3, col=1)
fig.update_yaxes(title_text='Wraps', row=4, col=1)
fig.update_xaxes(title_text='Time [s]', row=4, col=1, rangeslider=dict(visible=True))
fig.update_layout(height=1100, hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0))
fig.show()

In [38]:
period_info = estimate_period_seconds(
    processed_df['time_s'].to_numpy(dtype=float),
    processed_df['corrected_counts'].to_numpy(dtype=float),
    wrap_span=wrap_span_counts,
    min_rpm=period_search_min_rpm,
    max_rpm=period_search_max_rpm,
    sample_rate_hz_override=sample_rate_hz_override,
)

data_estimated_period_s = period_info['period_s']
data_estimated_frequency_hz = 1.0 / data_estimated_period_s
data_estimated_speed_rpm = 60.0 * data_estimated_frequency_hz

if driving_speed_rpm_override is None:
    selected_period_source = 'data_estimate'
    estimated_speed_rpm = data_estimated_speed_rpm
    estimated_frequency_hz = data_estimated_frequency_hz
    estimated_period_s = data_estimated_period_s
else:
    estimated_speed_rpm = float(driving_speed_rpm_override)
    if not np.isfinite(estimated_speed_rpm) or estimated_speed_rpm <= 0.0:
        raise ValueError('driving_speed_rpm_override must be positive when provided.')
    selected_period_source = 'driving_speed_rpm_override'
    estimated_frequency_hz = estimated_speed_rpm / 60.0
    estimated_period_s = 1.0 / estimated_frequency_hz

print(f"Estimated sample rate: {period_info['sample_rate_hz']:.3f} Hz")
print(f"Data-derived period: {data_estimated_period_s:.6f} s")
print(f"Data-derived frequency: {data_estimated_frequency_hz:.6f} Hz")
print(f"Data-derived crank speed: {data_estimated_speed_rpm:.3f} RPM")
print(f"Selected timing source: {selected_period_source}")
print(f"Selected period: {estimated_period_s:.6f} s")
print(f"Selected frequency: {estimated_frequency_hz:.6f} Hz")
print(f"Selected crank speed: {estimated_speed_rpm:.3f} RPM")
print(f"Period estimate method: {period_info['method']}")
print(f"Welch cross-check frequency: {period_info['dominant_frequency_hz']:.6f} Hz")
print(f"Candidate periods [s]: {np.array2string(period_info['candidate_periods_s'], precision=6)}")

Estimated sample rate: 500.000 Hz
Data-derived period: 0.112000 s
Data-derived frequency: 8.928571 Hz
Data-derived crank speed: 535.714 RPM
Selected timing source: data_estimate
Selected period: 0.112000 s
Selected frequency: 8.928571 Hz
Selected crank speed: 535.714 RPM
Period estimate method: circular-autocorrelation
Welch cross-check frequency: 8.997001 Hz
Candidate periods [s]: [0.112 0.086 0.056]


In [39]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=('Wrapped signal used for period estimation', 'Circular autocorrelation period search'),
)
fig.add_trace(
    go.Scatter(x=period_info['t_valid'], y=period_info['wrapped_counts_valid'], mode='lines', name='Wrapped corrected counts', line=dict(width=1.0)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=period_info['lag_s'], y=period_info['autocorr'], mode='lines', name='Circular autocorrelation', line=dict(width=1.2)),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=period_info['lag_s'][period_info['search_mask']],
        y=period_info['autocorr'][period_info['search_mask']],
        mode='lines',
        name='Search window',
        line=dict(width=2.0, color='orange'),
    ),
    row=2,
    col=1,
)
fig.add_vline(x=period_info['period_s'], line_dash='dash', line_color='red', row=2, col=1)
fig.update_yaxes(title_text='Counts', row=1, col=1)
fig.update_yaxes(title_text='Autocorrelation', row=2, col=1)
fig.update_xaxes(title_text='Time [s]', row=1, col=1, rangeslider=dict(visible=True))
fig.update_xaxes(title_text='Lag [s]', row=2, col=1, rangeslider=dict(visible=True))
fig.update_layout(height=800, hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0))
fig.show()

In [40]:
fit_info = fit_scale_and_shift(
    processed_df['time_s'].to_numpy(dtype=float),
    processed_df['unwrapped_counts_zeroed'].to_numpy(dtype=float),
    r_mm=crank_pin_offset_r_mm,
    l_mm=entry_offset_l_mm,
    period_guess_s=estimated_period_s,
    shift_fraction=fit_shift_search_fraction_of_period,
    min_valid_fraction=minimum_valid_fraction_for_fit,
    sample_rate_hz=float(period_info['sample_rate_hz']),
)

analysis_df = processed_df.copy()
analysis_df['ideal_displacement_mm'] = fit_info['ideal_displacement_mm']
analysis_df['shifted_unwrapped_counts'] = fit_info['shifted_counts']
analysis_df['scaled_measured_mm'] = fit_info['scaled_series_mm']
analysis_df['residual_mm'] = fit_info['residuals_mm']
analysis_df['fit_valid'] = fit_info['valid_mask']

fit_info['analysis_df'] = analysis_df

print(f"Initial period guess: {fit_info['period_initial_guess_s']:.6f} s")
print(f"Interpolated cycle-mean period: {fit_info['refined_cycle_period_s']:.6f} s")
print(f"Best-fit period: {fit_info['period_s']:.6f} s")
print(f"Best-fit speed: {fit_info['speed_rpm']:.3f} RPM")
print(f"Best-fit time shift: {fit_info['shift_s']:.6f} s")
print(f"Period/shift search method: {fit_info['period_shift_search_method']}")
print(f"Extrema-alignment RMSE: {fit_info['extrema_alignment_rmse_s'] * 1e3:.3f} ms")
print(f"Best-fit RMSE: {fit_info['rmse_mm']:.6f} mm")

Initial period guess: 0.112000 s
Interpolated cycle-mean period: 0.112039 s
Best-fit period: 0.112042 s
Best-fit speed: 535.515 RPM
Best-fit time shift: 0.051322 s
Period/shift search method: global_grid_then_local_refine
Extrema-alignment RMSE: 0.136 ms
Best-fit RMSE: 0.453536 mm


In [41]:
cycle_peak_signal = analysis_df['unwrapped_counts_zeroed'].to_numpy(dtype=float)
cycle_peak_time_s = analysis_df['time_s'].to_numpy(dtype=float)
cycle_peak_extrema = detect_principal_extrema_times(
    cycle_peak_time_s,
    cycle_peak_signal,
    period_s=fit_info['period_s'],
    sample_rate_hz=float(period_info['sample_rate_hz']),
)

cycle_peak_times_s = np.asarray(cycle_peak_extrema['peak_times_s'], dtype=float)
cycle_peak_values = np.asarray(cycle_peak_extrema['peak_values'], dtype=float)
cycle_peak_times_sample_s = np.asarray(cycle_peak_extrema['peak_times_sample_s'], dtype=float)

if cycle_peak_times_s.size < 2:
    raise ValueError('Could not identify at least two peaks in the unwrapped waveform for cycle timing analysis.')

cycle_peak_to_peak_s = np.diff(cycle_peak_times_s)
cycle_peak_mid_times_s = 0.5 * (cycle_peak_times_s[:-1] + cycle_peak_times_s[1:])
cycle_peak_to_peak_sample_s = np.diff(cycle_peak_times_sample_s)

cycle_interval_df = pd.DataFrame(
    {
        'cycle_mid_time_s': cycle_peak_mid_times_s,
        'peak_to_peak_time_s': cycle_peak_to_peak_s,
        'sample_snapped_peak_to_peak_time_s': cycle_peak_to_peak_sample_s,
    }
)

print(f"Detected waveform peaks: {len(cycle_peak_times_s)}")
print(f"Interpolated peak-to-peak mean: {np.mean(cycle_peak_to_peak_s):.6f} s")
print(f"Interpolated peak-to-peak std: {np.std(cycle_peak_to_peak_s):.6f} s")
print(f"Interpolated peak-to-peak min/max: {np.min(cycle_peak_to_peak_s):.6f} / {np.max(cycle_peak_to_peak_s):.6f} s")
print(f"Sample-snapped peak-to-peak std: {np.std(cycle_peak_to_peak_sample_s):.6f} s")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.10,
    subplot_titles=('Cycle Peak-to-Peak Time Series', 'Cycle Peak-to-Peak Histogram'),
)
fig.add_trace(
    go.Scatter(
        x=cycle_peak_time_s,
        y=cycle_peak_signal,
        mode='lines',
        name='Unwrapped displacement [counts]',
        line=dict(width=1.0, color='rgba(30, 30, 30, 0.45)'),
        visible='legendonly',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=cycle_peak_times_s,
        y=cycle_peak_values,
        mode='markers',
        name='Detected peaks (interpolated)',
        marker=dict(symbol='diamond', size=6, color='royalblue'),
        visible='legendonly',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=cycle_peak_mid_times_s,
        y=cycle_peak_to_peak_sample_s,
        mode='lines+markers',
        name='Peak-to-peak time [s] sample-snapped',
        marker=dict(symbol='diamond', size=5, color='rgba(120, 120, 120, 0.75)'),
        line=dict(width=1.0, color='rgba(120, 120, 120, 0.75)'),
        visible='legendonly',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=cycle_peak_mid_times_s,
        y=cycle_peak_to_peak_s,
        mode='lines+markers',
        name='Peak-to-peak time [s] interpolated',
        marker=dict(symbol='diamond', size=6, color='crimson'),
        line=dict(width=1.5, color='crimson'),
    ),
    row=1,
    col=1,
)
fig.add_hline(y=float(np.mean(cycle_peak_to_peak_s)), line_dash='dash', line_color='gray', row=1, col=1)
fig.add_trace(
    go.Histogram(
        x=cycle_peak_to_peak_s,
        nbinsx=min(30, max(8, int(np.sqrt(len(cycle_peak_to_peak_s))))),
        name='Peak-to-peak histogram',
        marker=dict(color='seagreen'),
    ),
    row=2,
    col=1,
)
fig.update_yaxes(title_text='Peak-to-peak time [s]', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_xaxes(title_text='Cycle midpoint time [s]', row=1, col=1, rangeslider=dict(visible=True))
fig.update_xaxes(title_text='Peak-to-peak time [s]', row=2, col=1)
fig.update_layout(height=900, hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0))
fig.show()

Detected waveform peaks: 54
Interpolated peak-to-peak mean: 0.112040 s
Interpolated peak-to-peak std: 0.000193 s
Interpolated peak-to-peak min/max: 0.111389 / 0.112600 s
Sample-snapped peak-to-peak std: 0.000272 s


In [42]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=('Ideal vs fitted measured displacement', 'Fit residuals'),
)
fig.add_trace(
    go.Scatter(x=analysis_df['time_s'], y=analysis_df['ideal_displacement_mm'], mode='lines', name='Ideal displacement [mm]', line=dict(width=2.0)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=analysis_df['time_s'], y=analysis_df['scaled_measured_mm'], mode='lines', name='Scaled measured series [mm]', line=dict(width=1.2)),
    row=1,
    col=1,
)
if show_measured_displacement_datapoints:
    fig.add_trace(
        go.Scatter(
            x=analysis_df['time_s'],
            y=analysis_df['scaled_measured_mm'],
            mode='markers',
            name='Measured displacement datapoints [mm]',
            marker=dict(symbol='diamond', size=4, color='rgba(239, 85, 59, 0.85)'),
        ),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter(x=analysis_df['time_s'], y=analysis_df['residual_mm'], mode='lines', name='Residual [mm]', line=dict(width=1.0, color='red')),
    row=2,
    col=1,
)
fig.add_hline(y=0.0, line_width=0.8, line_color='black', row=2, col=1)
fig.update_yaxes(title_text='Displacement [mm]', row=1, col=1)
fig.update_yaxes(title_text='Residual [mm]', row=2, col=1)
fig.update_xaxes(title_text='Time [s]', row=2, col=1, rangeslider=dict(visible=True))
fig.update_layout(height=800, hovermode='x unified', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0))
fig.show()

In [43]:
phase_bin_df, phase_harmonics_df, phase_residual_info = summarize_residual_vs_phase(
    analysis_df['time_s'].to_numpy(dtype=float),
    analysis_df['residual_mm'].to_numpy(dtype=float),
    analysis_df['fit_valid'].to_numpy(dtype=bool),
    period_s=float(fit_info['period_s']),
    bin_count=int(phase_bin_count),
    max_harmonic=int(phase_harmonic_order),
)
fit_info['phase_residual_info'] = phase_residual_info

phase_plot_df = phase_bin_df.copy()
if len(phase_plot_df) > 0:
    phase_wrap_row = phase_plot_df.iloc[[0]].copy()
    phase_wrap_row['phase_center_deg'] = phase_wrap_row['phase_center_deg'] + 360.0
    phase_plot_df = pd.concat([phase_plot_df, phase_wrap_row], ignore_index=True)

print(f"Phase bins: {int(phase_residual_info['bin_count'])}")
print(
    "Phase-mean residual peak-to-peak: "
    f"{phase_bin_df['mean_residual_mm'].max() - phase_bin_df['mean_residual_mm'].min():.6f} mm"
)
print(f"Max phase-binned RMS residual: {phase_bin_df['rms_residual_mm'].max():.6f} mm")
display(phase_harmonics_df)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=('Mean Residual vs Phase', 'Residual Spread vs Phase', 'Samples per Phase Bin'),
)
fig.add_trace(
    go.Scatter(
        x=phase_plot_df['phase_center_deg'],
        y=phase_plot_df['mean_plus_std_mm'],
        mode='lines',
        name='Mean + 1 std [mm]',
        line=dict(width=0),
        hoverinfo='skip',
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=phase_plot_df['phase_center_deg'],
        y=phase_plot_df['mean_minus_std_mm'],
        mode='lines',
        name='Mean +/- 1 std [mm]',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(239, 85, 59, 0.15)',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=phase_plot_df['phase_center_deg'],
        y=phase_plot_df['mean_residual_mm'],
        mode='lines+markers',
        name='Mean residual [mm]',
        line=dict(width=2.0, color='crimson'),
        marker=dict(size=5, symbol='diamond'),
    ),
    row=1,
    col=1,
)
fig.add_hline(y=0.0, line_width=0.8, line_color='black', row=1, col=1)
fig.add_trace(
    go.Scatter(
        x=phase_plot_df['phase_center_deg'],
        y=phase_plot_df['rms_residual_mm'],
        mode='lines+markers',
        name='Residual RMS [mm]',
        line=dict(width=2.0, color='royalblue'),
        marker=dict(size=5, symbol='diamond'),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=phase_plot_df['phase_center_deg'],
        y=phase_plot_df['std_residual_mm'],
        mode='lines+markers',
        name='Residual std [mm]',
        line=dict(width=1.6, color='darkorange'),
        marker=dict(size=5, symbol='diamond'),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=phase_bin_df['phase_center_deg'],
        y=phase_bin_df['sample_count'],
        name='Samples per bin',
        marker=dict(color='seagreen'),
    ),
    row=3,
    col=1,
)
fig.update_yaxes(title_text='Residual [mm]', row=1, col=1)
fig.update_yaxes(title_text='Spread [mm]', row=2, col=1)
fig.update_yaxes(title_text='Samples', row=3, col=1)
fig.update_xaxes(title_text='Phase [deg]', row=3, col=1, range=[0.0, 360.0])
fig.update_layout(
    height=1000,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0),
)
fig.show()

Phase bins: 72
Phase-mean residual peak-to-peak: 1.338169 mm
Max phase-binned RMS residual: 0.917280 mm


,harmonic,amplitude_mm,phase_offset_deg,cos_coeff_mm,sin_coeff_mm
0,0,0.077950,NaN,0.077950,NaN
1,1,0.073557,55.696630,0.041455,0.060763
2,2,0.214387,8.946680,0.211779,0.033340
3,3,0.012008,269.973788,-0.000005,-0.012008


In [44]:
valid_residuals = analysis_df.loc[analysis_df['fit_valid'], 'residual_mm'].to_numpy(dtype=float)

summary = {
    'csv_path': str(csv_path),
    'time_column': time_column,
    'raw_adc_column': raw_adc_column,
    'analysis_start_time_s': None if analysis_start_time_s is None else float(analysis_start_time_s),
    'analysis_end_time_s': None if analysis_end_time_s is None else float(analysis_end_time_s),
    'driving_speed_rpm_override': None if driving_speed_rpm_override is None else float(driving_speed_rpm_override),
    'selected_period_source': selected_period_source,
    'fit_time_shift_objective': fit_info['shift_objective'],
    'extrema_alignment_rmse_ms': float(1e3 * fit_info['extrema_alignment_rmse_s']),
    'wrap_span_counts': float(wrap_span_counts),
    'poly_c0': float(c0),
    'poly_c1': float(c1),
    'poly_c2': float(c2),
    'poly_c3': float(c3),
    'geometry_r_mm': float(crank_pin_offset_r_mm),
    'geometry_l_mm': float(entry_offset_l_mm),
    'sample_count': int(len(analysis_df)),
    'valid_fit_samples': int(analysis_df['fit_valid'].sum()),
    'duration_s': float(np.nanmax(analysis_df['time_s']) - np.nanmin(analysis_df['time_s'])),
    'estimated_sample_rate_hz': float(period_info['sample_rate_hz']),
    'data_estimated_period_s': float(data_estimated_period_s),
    'data_estimated_frequency_hz': float(data_estimated_frequency_hz),
    'data_estimated_speed_rpm': float(data_estimated_speed_rpm),
    'estimated_period_s': float(estimated_period_s),
    'estimated_frequency_hz': float(estimated_frequency_hz),
    'estimated_speed_rpm': float(estimated_speed_rpm),
    'fit_period_initial_guess_s': float(fit_info['period_initial_guess_s']),
    'fit_period_cycle_mean_s': float(fit_info['refined_cycle_period_s']),
    'fit_period_cycle_std_s': float(fit_info['refined_cycle_period_std_s']),
    'fit_period_s': float(fit_info['period_s']),
    'fit_frequency_hz': float(fit_info['frequency_hz']),
    'fit_speed_rpm': float(fit_info['speed_rpm']),
    'fit_scale_mm_per_count': float(fit_info['scale_mm_per_count']),
    'fit_time_shift_s': float(fit_info['shift_s']),
    'peak_matched_measured_peak_count': float(fit_info['measured_peak_count']),
    'peak_matched_ideal_peak_mm': float(fit_info['ideal_peak_mm']),
    'peak_matched_scaled_peak_mm': float(np.nanmax(analysis_df.loc[analysis_df['fit_valid'], 'scaled_measured_mm'].to_numpy(dtype=float))),
    'rmse_mm': float(np.sqrt(np.nanmean(valid_residuals**2))),
    'mae_mm': float(np.nanmean(np.abs(valid_residuals))),
    'max_abs_residual_mm': float(np.nanmax(np.abs(valid_residuals))),
    'phase_bin_count': int(phase_residual_info['bin_count']),
    'phase_mean_residual_peak_to_peak_mm': float(
        phase_bin_df['mean_residual_mm'].max() - phase_bin_df['mean_residual_mm'].min()
    ),
    'phase_max_rms_residual_mm': float(phase_bin_df['rms_residual_mm'].max()),
    'phase_mean_std_residual_mm': float(phase_bin_df['std_residual_mm'].mean()),
    'raw_min_count': float(np.nanmin(analysis_df['raw_adc_counts'])),
    'raw_max_count': float(np.nanmax(analysis_df['raw_adc_counts'])),
    'unwrapped_min_count': float(np.nanmin(analysis_df['unwrapped_counts'])),
    'unwrapped_max_count': float(np.nanmax(analysis_df['unwrapped_counts'])),
    'ideal_min_mm': float(np.nanmin(analysis_df['ideal_displacement_mm'])),
    'ideal_max_mm': float(np.nanmax(analysis_df['ideal_displacement_mm'])),
}

for _, harmonic_row in phase_harmonics_df.iterrows():
    harmonic_idx = int(harmonic_row['harmonic'])
    summary[f'phase_h{harmonic_idx}_amplitude_mm'] = float(harmonic_row['amplitude_mm'])
    if harmonic_idx > 0:
        summary[f'phase_h{harmonic_idx}_phase_deg'] = float(harmonic_row['phase_offset_deg'])

summary_df = pd.DataFrame({'metric': list(summary.keys()), 'value': list(summary.values())})
display(summary_df)

,metric,value
0,csv_path,500rpm.CSV
1,time_column,timestamp
2,raw_adc_column,i2c_stringpot_raw [counts]
3,analysis_start_time_s,2.0
4,analysis_end_time_s,8.0
5,driving_speed_rpm_override,None
6,selected_period_source,data_estimate
7,fit_time_shift_objective,extrema_alignment_period_and_shift
8,extrema_alignment_rmse_ms,0.136336
9,wrap_span_counts,4095.0


## Notes

- The polynomial is applied directly to the raw ADC counts and returns corrected counts.
- `analysis_start_time_s` and `analysis_end_time_s` define the inclusive time window, in seconds from the start of the file, used by the notebook.
- `driving_speed_rpm_override` is optional. Leave it as `None` to use the notebook's current data-driven period estimate; set it to a positive RPM value to force the ideal-profile timing used for fitting.
- Unwrapping now uses the sign of the recent travel direction together with configurable low and high wrap bands. A wrap is only declared when the signal crosses from the high band to the low band while moving upward, or from the low band to the high band while moving downward.
- The fit uses the measured series shifted in time and scaled in amplitude only. Before fitting, the unwrapped counts are zeroed to their minimum so the measured series can be compared against the ideal displacement profile, which is also referenced to its minimum string length. The scale is peak-matched so the fitted measured peak displacement equals the ideal peak displacement exactly over the valid fit window.
- Period estimation is based on circular autocorrelation of the wrapped corrected counts, with an RPM-bounded search window and a Welch-spectrum cross-check.